# Week 3 · Regional Root-Cause Analysis & High-Risk Corridors
**E-Commerce Delivery & Logistics Analysis** · Owner: Akhila Nair · Branch: `feature/akhila-regional-rootcause` · Issue #11

**Mission:** investigate why deliveries to **Rio de Janeiro (RJ)** and **Bahia (BA)** are slower and later than average, separate road/last-mile effects from warehouse effects, understand how freight cost scales with distance, and estimate the impact of regional micro-fulfilment hubs.

**Analytical standards applied**
- Every percentage is shown with its order count, and groups below **20 orders** are excluded.
- Findings are described as *associations*; causes are not asserted.
- Each section ends with a simulation or a quantified operational implication ("how many delayed orders could be saved?").

## 1. Setup & data loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.options.display.float_format = "{:,.2f}".format

# --- Project paths (works whether the notebook is run from /notebooks or the repo root) ---
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"          # read-only: never modified
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Analysis parameters ---
MIN_ORDERS = 20                                   # minimum sample size for any group comparison
PRIMARY, ACCENT, NEUTRAL = "#1F4E79", "#C0392B", "#9AA5B1"

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two sets of coordinates."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def add_distance_km(frame):
    """Add seller->customer distance (km) using the mean coordinates of each ZIP prefix.

    The analytical dataset has no coordinates, so they are derived from the raw geolocation and
    sellers tables (read-only). Distance is therefore an approximation at ZIP-prefix level.
    """
    geo = pd.read_csv(DATA_RAW / "olist_geolocation_dataset.csv")
    centroid = geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()
    sellers = pd.read_csv(DATA_RAW / "olist_sellers_dataset.csv")[["seller_id", "seller_zip_code_prefix"]]

    out = frame.merge(sellers, on="seller_id", how="left")
    out = out.merge(centroid.add_prefix("cust_"), left_on="customer_zip_code_prefix", right_index=True, how="left")
    out = out.merge(centroid.add_prefix("sell_"), left_on="seller_zip_code_prefix", right_index=True, how="left")
    out["distance_km"] = haversine_km(out["sell_geolocation_lat"], out["sell_geolocation_lng"],
                                      out["cust_geolocation_lat"], out["cust_geolocation_lng"])
    return out.drop(columns=[c for c in out.columns if c.startswith(("cust_geo", "sell_geo"))])


BAND_EDGES = [0, 100, 300, 600, 1000, 1500, 2000, np.inf]
BAND_LABELS = ["<100 km", "100-300 km", "300-600 km", "600-1,000 km", "1,000-1,500 km", "1,500-2,000 km", ">2,000 km"]


def delay_rate(x):
    """Share of orders delivered late, in %."""
    return (x == "delayed").mean() * 100

In [ ]:
processed_path = PROJECT_ROOT / "data" / "processed" / "validated_kpi_dataset.csv"
interim_path = PROJECT_ROOT / "data" / "interim" / "logistics_analytical_dataset.csv"

# Week 3 guide: use the validated KPI dataset produced by Aman. Fall back to the interim dataset
# (same schema) until his branch has been merged and the file has been generated locally.
source_path = processed_path if processed_path.exists() else interim_path
print("Data source:", source_path.relative_to(PROJECT_ROOT))

df = pd.read_csv(source_path)
valid = df[df["is_eligible_otd"] == 1].copy()
valid = add_distance_km(valid)

national_delay = delay_rate(valid["delivery_outcome"])
total_delayed = (valid["delivery_outcome"] == "delayed").sum()
print(f"Eligible orders: {len(valid):,} | delayed: {total_delayed:,} | national delay rate: {national_delay:.2f}%")
print(f"Orders without computable distance: {valid['distance_km'].isna().sum():,} (excluded from distance-based cells only)")

NORTHEAST = ["MA", "PI", "CE", "RN", "PB", "PE", "AL", "SE", "BA"]
SOUTHEAST_HUB = ["RJ", "ES"]          # Southeast states above the national delay rate

## 2. Profile of the two high-risk destinations (Day 1)
RJ and BA are compared with the national baseline and with two reference states: São Paulo (SP, the main seller hub) and Minas Gerais (MG, a Southeast neighbour of similar size).

In [ ]:
def state_profile(frame, label):
    late = frame[frame["delivery_outcome"] == "delayed"]
    return pd.Series({
        "orders": len(frame),
        "delayed_orders": len(late),
        "delay_rate_pct": delay_rate(frame["delivery_outcome"]),
        "share_of_all_delays_pct": len(late) / total_delayed * 100,
        "avg_days_late_when_late": late["delay_days"].mean(),
        "median_promised_days": frame["promised_delivery_duration_days"].median(),
        "median_actual_days": frame["total_delivery_duration_days"].median(),
        "median_transit_days": frame["carrier_transit_time_days"].median(),
        "avg_rating": frame["review_score"].mean(),
        "pct_1star": (frame["review_score"].dropna() == 1).mean() * 100,
    }, name=label)

rows = [state_profile(valid[valid["customer_state"] == s], s) for s in ["RJ", "BA", "MG", "SP"]]
rows.append(state_profile(valid, "National"))
profile = pd.DataFrame(rows)
display(profile)

**Inbound corridors.** Which origin states feed RJ and BA, and which of those corridors are late most often (≥ 20 orders)?

In [ ]:
inbound = (valid[valid["customer_state"].isin(["RJ", "BA"])]
           .groupby(["customer_state", "seller_state"])
           .agg(orders=("order_id", "count"),
                delayed_orders=("delivery_outcome", lambda x: (x == "delayed").sum()),
                delay_rate_pct=("delivery_outcome", delay_rate),
                median_transit_days=("carrier_transit_time_days", "median"))
           .reset_index())
inbound = inbound[inbound["orders"] >= MIN_ORDERS].sort_values(["customer_state", "orders"], ascending=[True, False])
display(inbound.groupby("customer_state").head(6).reset_index(drop=True))

sp_share = {s: valid[(valid["customer_state"] == s) & (valid["seller_state"] == "SP")].shape[0] / (valid["customer_state"] == s).sum() * 100
            for s in ["RJ", "BA"]}
print(f"Share of RJ orders shipped from SP sellers: {sp_share['RJ']:.1f}% | BA: {sp_share['BA']:.1f}%")

## 3. Isolating road vs. warehouse delay with identical origin sellers (Day 2)
Restricting to **SP sellers** holds the origin warehouse region constant. If processing time is similar across destinations but transit time differs, the gap sits in the carrier network rather than in seller packaging.

In [ ]:
sp_sellers = valid[valid["seller_state"] == "SP"]
compare_states = ["SP", "MG", "RJ", "BA"]
corridors = (sp_sellers[sp_sellers["customer_state"].isin(compare_states)]
             .groupby("customer_state")
             .agg(order_volume=("order_id", "count"),
                  delayed_orders=("delivery_outcome", lambda x: (x == "delayed").sum()),
                  delay_rate_pct=("delivery_outcome", delay_rate),
                  median_distance_km=("distance_km", "median"),
                  median_processing_days=("order_processing_time_days", "median"),
                  median_transit_days=("carrier_transit_time_days", "median"),
                  median_promised_days=("promised_delivery_duration_days", "median"),
                  avg_freight=("total_freight_value", "mean"))
             .reindex(compare_states))
corridors["transit_days_per_100km"] = corridors["median_transit_days"] / corridors["median_distance_km"] * 100
display(corridors)

**Like-for-like check.** Within the same distance band (300–600 km from the seller), do RJ orders still lag other destinations? Destinations with < 20 orders are dropped.

In [ ]:
band_mask = sp_sellers["distance_km"].between(300, 600, inclusive="left")
like_for_like = (sp_sellers[band_mask]
                 .groupby("customer_state")
                 .agg(orders=("order_id", "count"),
                      delayed_orders=("delivery_outcome", lambda x: (x == "delayed").sum()),
                      delay_rate_pct=("delivery_outcome", delay_rate),
                      median_distance_km=("distance_km", "median"),
                      median_transit_days=("carrier_transit_time_days", "median"),
                      median_promised_days=("promised_delivery_duration_days", "median")))
like_for_like = like_for_like[like_for_like["orders"] >= MIN_ORDERS].sort_values("delay_rate_pct", ascending=False)
display(like_for_like)

## 4. Freight cost elasticity across distance bands (Day 3)
Freight is modelled as log-log in distance and weight, so each coefficient reads as an elasticity (the % change in freight for a 1% change in the driver). This describes how price scales; it does not prove how carriers set prices.

In [ ]:
geo = valid.dropna(subset=["distance_km", "total_freight_value", "total_weight_g"]).copy()
geo = geo[(geo["distance_km"] > 0) & (geo["total_freight_value"] > 0) & (geo["total_weight_g"] > 0)]
geo["distance_band"] = pd.cut(geo["distance_km"], bins=BAND_EDGES, labels=BAND_LABELS, right=False)

band_freight = geo.groupby("distance_band", observed=True).agg(
    orders=("order_id", "count"),
    avg_freight=("total_freight_value", "mean"),
    median_weight_g=("total_weight_g", "median"),
    median_transit_days=("carrier_transit_time_days", "median"),
    delay_rate_pct=("delivery_outcome", delay_rate),
)
band_freight["freight_vs_first_band"] = band_freight["avg_freight"] / band_freight["avg_freight"].iloc[0]
display(band_freight)

X = np.column_stack([np.ones(len(geo)), np.log(geo["distance_km"]), np.log(geo["total_weight_g"])])
y = np.log(geo["total_freight_value"])
coef, *_ = np.linalg.lstsq(X, y, rcond=None)
r2 = 1 - ((y - X @ coef) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"Distance elasticity of freight : {coef[1]:.3f}")
print(f"Weight elasticity of freight   : {coef[2]:.3f}")
print(f"R-squared                      : {r2:.3f}   (n = {len(geo):,})")

In [ ]:
# Do RJ / BA pay more than distance and weight alone would predict?
geo["freight_residual"] = y - X @ coef                      # log-points above/below model expectation
resid = (geo[geo["customer_state"].isin(["SP", "MG", "RJ", "BA", "PE", "CE"])]
         .groupby("customer_state")
         .agg(orders=("order_id", "count"),
              avg_freight=("total_freight_value", "mean"),
              mean_log_residual=("freight_residual", "mean")))
resid["freight_vs_model_pct"] = (np.exp(resid["mean_log_residual"]) - 1) * 100
display(resid.sort_values("freight_vs_model_pct", ascending=False))

## 5. Micro-fulfilment hub simulation: Southeast vs. Northeast (Day 4)
**Scenario logic.** A regional hub is assumed to shorten the carrier leg by *X* days for every order into the target states. An order that was late by *d* days is rescued if *d ≤ X*. Orders already on time stay on time.

This is a **theoretical upper bound**: it assumes the saving is achieved on every order and ignores hub cost, inventory duplication and seller adoption.

In [ ]:
def simulate_hub(frame, states, days_saved):
    sub = frame[frame["customer_state"].isin(states)]
    late = sub[sub["delivery_outcome"] == "delayed"]
    rescued = (late["delay_days"] <= days_saved).sum()
    return {"orders": len(sub), "delayed_before": len(late),
            "rescued": int(rescued), "delayed_after": len(late) - int(rescued)}

scenarios = {"Southeast hub (RJ, ES)": SOUTHEAST_HUB, "Northeast hub (9 NE states)": NORTHEAST}
records = []
for name, states in scenarios.items():
    for days in [1, 2, 3, 5]:
        r = simulate_hub(valid, states, days)
        records.append({"scenario": name, "days_saved": days, **r,
                        "delay_rate_before_pct": r["delayed_before"] / r["orders"] * 100,
                        "delay_rate_after_pct": r["delayed_after"] / r["orders"] * 100,
                        "pct_of_region_delays_rescued": r["rescued"] / r["delayed_before"] * 100,
                        "pct_of_national_delays_rescued": r["rescued"] / total_delayed * 100})
sim = pd.DataFrame(records)
display(sim)

In [ ]:
# Satisfaction context (association only): ratings of delayed vs on-time orders in the target regions
for name, states in scenarios.items():
    sub = valid[valid["customer_state"].isin(states)].dropna(subset=["review_score"])
    by = sub.groupby("delivery_outcome")["review_score"].agg(["count", "mean", lambda x: (x == 1).mean() * 100])
    by.columns = ["reviews", "avg_rating", "pct_1star"]
    print(name); display(by)

## 6. Summary figure (Day 5)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 12))

# (A) delay rate for SP-origin orders by destination
a = corridors.reset_index()
ax = axes[0, 0]
bars = ax.bar(a["customer_state"], a["delay_rate_pct"], color=[NEUTRAL, NEUTRAL, ACCENT, ACCENT])
for b, n in zip(bars, a["order_volume"]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3, f"{b.get_height():.1f}%\nn={n:,}", ha="center", fontsize=9)
ax.set_title("A. Delay rate, SP-origin sellers only", fontweight="bold"); ax.grid(axis="x", visible=False)
ax.set_ylabel("Delay rate (%)"); ax.set_ylim(0, a["delay_rate_pct"].max() * 1.3)

# (B) processing vs transit
ax = axes[0, 1]
idx = np.arange(len(a)); w = 0.38
ax.bar(idx - w / 2, a["median_processing_days"], w, label="Seller processing", color=NEUTRAL)
ax.bar(idx + w / 2, a["median_transit_days"], w, label="Carrier transit", color=PRIMARY)
ax.set_xticks(idx); ax.set_xticklabels(a["customer_state"])
ax.set_title("B. Median days: warehouse vs. carrier (SP sellers)", fontweight="bold")
ax.set_ylabel("Days"); ax.legend(); ax.grid(axis="x", visible=False)

# (C) distance vs transit by destination state
state_xy = (valid.groupby("customer_state")
            .agg(orders=("order_id", "count"), km=("distance_km", "median"),
                 transit=("carrier_transit_time_days", "median"))
            .query("orders >= @MIN_ORDERS"))
ax = axes[1, 0]
hi = state_xy.index.isin(["RJ", "BA"])
ax.scatter(state_xy.loc[~hi, "km"], state_xy.loc[~hi, "transit"], s=state_xy.loc[~hi, "orders"] / 25 + 20, color=NEUTRAL, alpha=0.8)
ax.scatter(state_xy.loc[hi, "km"], state_xy.loc[hi, "transit"], s=state_xy.loc[hi, "orders"] / 25 + 20, color=ACCENT, alpha=0.9)
for s, r in state_xy.iterrows():
    if s in ["RJ", "BA", "SP", "MG", "AL", "MA", "AM", "PR", "ES"]:
        dy = -13 if s in ["MG", "ES"] else 5
        ax.annotate(s, (r["km"], r["transit"]), xytext=(6, dy), textcoords="offset points", fontsize=9)
ax.set_title("C. Median distance vs. median carrier transit (bubble = orders)", fontweight="bold")
ax.set_xlabel("Median seller→customer distance (km)"); ax.set_ylabel("Median transit days")

# (D) hub simulation
ax = axes[1, 1]
for (name, color), dy in zip(zip(scenarios, [PRIMARY, ACCENT]), [-16, 9]):
    s = sim[sim["scenario"] == name]
    ax.plot(s["days_saved"], s["pct_of_region_delays_rescued"], marker="o", linewidth=2.2, color=color, label=name)
    for _, r in s.iterrows():
        ax.annotate(f"{int(r['rescued']):,}", (r["days_saved"], r["pct_of_region_delays_rescued"]), xytext=(0, dy), textcoords="offset points", ha="center", fontsize=8.5, color=color)
ax.set_title("D. Delayed orders rescued by a regional hub (labels = orders)", fontweight="bold")
ax.set_xlabel("Transit days saved per order"); ax.set_ylabel("% of region's delayed orders rescued"); ax.set_xticks([1, 2, 3, 5]); ax.legend()

fig.suptitle("High-risk corridor deep dive: Rio de Janeiro and Bahia", fontsize=16, fontweight="bold", y=0.995)
plt.tight_layout()
out = FIG_DIR / "high_risk_corridors_deepdive.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out, "| exists:", out.exists())

## 7. Findings, operational implications & caveats

All figures below come from the cells above (eligible delivered orders, ≥ 20 orders per group). Wording is deliberately associative.

**Findings**

1. **RJ and BA are over-represented in late deliveries.** RJ has 12,330 orders with 1,664 delayed (13.5% vs 8.1% nationally) and accounts for about 21% of all late orders; BA has 3,252 orders with 456 delayed (14.0%).
2. **The gap appears in carrier transit, not in seller processing.** For SP-origin sellers, median processing time is almost identical across destinations (2.2–2.4 days), while median transit grows from 3.6 days (SP) to 7.3 (MG), 9.1 (RJ) and 14.1 (BA). Delay rates are 15.5% for RJ (n=8,143) and 15.0% for BA (n=2,305) against 6.2–6.4% for SP and MG.
3. **RJ is slower than its distance alone would suggest.** In the 300–600 km band, SP-origin orders to RJ are late 15.0% of the time (n=6,572) against 6.5% for MG (n=5,018) and 4.9% for PR, with roughly 1.7 more transit days than MG at a similar promised window. This is consistent with a last-mile / regional network issue; it does not by itself identify the cause (congestion, carrier capacity and risk-area handling are all candidates).
4. **BA is largely a distance-and-promise story.** Its median distance from SP sellers is about 1,450 km, transit is about 14 days, and its promised window (about 29 days) is already generous, yet 15% of orders still arrive late.
5. **Freight does not explain the gap.** Freight scales weakly with distance (elasticity about 0.20) and weight (about 0.22), R² = 0.56. RJ and BA pay only about 3% and 2% above the model expectation, so customers in RJ pay roughly the same as comparable-distance MG customers while experiencing more than twice the delay rate.
6. **Delays are costly for satisfaction (association).** In RJ/ES, delayed orders average a 2.2 rating with 56% 1-star reviews, versus 4.25 and 8% for on-time orders. Product quality and other confounders are not controlled, so this is not proof that lateness alone causes the low scores.

**Simulated impact of regional micro-fulfilment hubs (theoretical upper bound)**

| Scenario | Orders | Delayed today | Rescued if 2 days saved | Rescued if 3 days saved |
|---|---|---|---|---|
| Southeast hub (RJ, ES) | 14,319 | 1,908 | 349 (18.3%) | 458 (24.0%) |
| Northeast hub (9 NE states) | 9,031 | 1,295 | 250 (19.3%) | 326 (25.2%) |

Late orders in RJ are late by about 12.9 days on average, so a 2–3 day saving rescues only about a quarter of them. A hub alone is unlikely to fix the problem.

**Recommended actions**
- Open a **carrier SLA review for SP→RJ** (the highest-volume high-risk corridor: 8,143 orders, 1,264 delayed) with transit-time targets and per-carrier tracking.
- **Re-calibrate promised delivery dates** for RJ-bound and Northeast orders so that estimates reflect observed transit, in parallel with any hub investment.
- **Pilot a Northeast hub or forward stock** for the highest-volume NE destination (BA) and measure transit days saved before scaling.
- Add **carrier identity** to the dataset so RJ's slowdown can be attributed to specific carriers.

**Caveats**
- Distance is approximated from ZIP-prefix centroids (great-circle, not road distance); 478 orders (0.5%) could not be geocoded.
- Carrier and seller-specific effects are not observable in this dataset, so root causes are hypotheses to test, not conclusions.
- Hub scenarios assume the full time saving on every order and ignore cost, inventory duplication and seller adoption.
- Delay is measured against the *promised* date, so regions with generous promises can look better than their actual speed.
- The notebook reads `data/processed/validated_kpi_dataset.csv` when present and otherwise falls back to the interim dataset; re-run after Aman's file is merged and confirm the figures are unchanged.